In [11]:
import pandas as pd
import numpy as np

import featuretools as ft

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, mutual_info_regression
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, mutual_info_regression
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE, mutual_info_regression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

OUR TARGET VARIABLE IS Charges

VARIABLE COMBINATIONS

Deep Feature Synthesis (DFS)------Generating New Variables

In [12]:
# Load the data
df= pd.read_csv("insurance.csv")
df.head()


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [13]:
df.shape


(1338, 7)

In [14]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [15]:
# No of categories in region
df['region'].nunique()

4

In [16]:
# unique categories in region
df['region'].unique()

array(['southwest', 'southeast', 'northwest', 'northeast'], dtype=object)

In [17]:
# Converts categorical columns to numerical in sex, region, smoker
df['sex'] = df['sex'].map({'male': 1, 'female': 0})
df['smoker'] = df['smoker'].map({'yes': 1, 'no': 0})

# Onehot encoding for region
df = pd.get_dummies(df, columns=['region'], drop_first=True)
df.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,False,False,True
1,18,1,33.770,1,0,1725.55230,False,True,False
2,28,1,33.000,3,0,4449.46200,False,True,False
3,33,1,22.705,0,0,21984.47061,True,False,False
4,32,1,28.880,0,0,3866.85520,True,False,False


In [18]:
# Remove NaN values
df = df.dropna()

# Remove duplicates
df = df.drop_duplicates()

# Reset the index
df = df.reset_index(drop=True)

In [19]:
# Target variable
target = "charges"

y = df[target]

# create an entity set with Insurance data
es = ft.EntitySet(id="Insurance Charge")
es = es.add_dataframe(dataframe_name="insurance_data", dataframe=df.drop(target, axis=1), index="index", make_index=True)

# Use DFS to generate new features
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="insurance_data",
    
    trans_primitives=["multiply_numeric"],
    
    max_depth=1 
)

feature_matrix.replace([np.inf, -np.inf], np.nan, inplace=True)
feature_matrix.dropna(axis=1, inplace=True)

# re-attach y back to feature_matrix
feature_matrix[target] = y

feature_matrix

,age,sex,bmi,children,smoker,region_northwest,region_southeast,region_southwest,age * bmi,age * children,age * sex,age * smoker,bmi * children,bmi * sex,bmi * smoker,children * sex,children * smoker,sex * smoker,charges
index,,,,,,,,,,,,,,,,,,,
0,19,0,27.900,0,1,False,False,True,530.100,0.0,0.0,19.0,0.00,0.000,27.90,0.0,0.0,0.0,16884.92400
1,18,1,33.770,1,0,False,True,False,607.860,18.0,18.0,0.0,33.77,33.770,0.00,1.0,0.0,0.0,1725.55230
2,28,1,33.000,3,0,False,True,False,924.000,84.0,28.0,0.0,99.00,33.000,0.00,3.0,0.0,0.0,4449.46200
3,33,1,22.705,0,0,True,False,False,749.265,0.0,33.0,0.0,0.00,22.705,0.00,0.0,0.0,0.0,21984.47061
4,32,1,28.880,0,0,True,False,False,924.160,0.0,32.0,0.0,0.00,28.880,0.00,0.0,0.0,0.0,3866.85520
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1332,50,1,30.970,3,0,True,False,False,1548.500,150.0,50.0,0.0,92.91,30.970,0.00,3.0,0.0,0.0,10600.54830
1333,18,0,31.920,0,0,False,False,False,574.560,0.0,0.0,0.0,0.00,0.000,0.00,0.0,0.0,0.0,2205.98080
1334,18,0,36.850,0,0,False,True,False,663.300,0.0,0.0,0.0,0.00,0.000,0.00,0.0,0.0,0.0,1629.83350


USING RFE TO SELECT BEST FEATURES

Linear Regressor

In [20]:
# X/y -split
X = feature_matrix.drop("charges", axis=1)
y = feature_matrix['charges']

# Use LinearRegressor to get important features
model = LinearRegression()

# create RFE 
rfe = RFE(estimator=model, n_features_to_select=10)

# fit the RFE model
rfe.fit(X, y)

# Ranking of the features
rankings = rfe.ranking_
support = rfe.support_

# Making DF to show results of RFE
results_df = pd.DataFrame({
    "Feature": X.columns,
    "Ranking": rankings,
    "Selected": support
}).sort_values(by="Ranking")

results_df

,Feature,Ranking,Selected
0,age,1,True
1,sex,1,True
15,children * sex,1,True
3,children,1,True
4,smoker,1,True
5,region_northwest,1,True
6,region_southeast,1,True
7,region_southwest,1,True
16,children * smoker,1,True
14,bmi * smoker,1,True
